In [ ]:
!pip install  decord

In [4]:
# ============================================================
#  Imports
# ============================================================

import os
import glob
import random
from dataclasses import dataclass
from typing import List, Dict, Any

import torch
import numpy as np
# ---------------------------
# Reproducibility
# ---------------------------
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

@dataclass
class CFG:
    # PATH TO YOUR RWF-2000 DATASET ON KAGGLE
    # Example structures:
    #   /kaggle/input/rwf-2000/RWF-2000/train/Fight/...
    #   /kaggle/input/rwf-2000/train/Fight/...
    #
    # Set this to the folder that directly contains `train/` and `val/`
    data_root: str = "/kaggle/input/rwf-2000/RWF-2000"  # <-- ADJUST IF NEEDED

    model_name: str = "MCG-NJU/videomae-base"
    num_frames: int = 16           # VideoMAE default
    frame_sample_rate: int = 4     # stride between sampled frames
    image_size: int = 224

    train_batch_size: int = 4
    val_batch_size: int = 4
    # Kaggle usually handles 2–4 workers fine; 2 is safer if you hit dataloader issues
    num_workers: int = 2

    num_epochs: int = 10
    learning_rate: float = 5e-5
    weight_decay: float = 1e-4
    warmup_ratio: float = 0.1

    mixed_precision: bool = True   # use AMP on GPU

    output_dir: str = "./checkpoints_videomae_rwf2000"
    project_name: str = "rwf2000-video-violence"
    run_name: str = "videomae-base-rwf2000"

    seed: int = 42

set_seed(CFG.seed)

os.makedirs(CFG.output_dir, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cuda


In [ ]:
# ============================================================
# 2. Weights & Biases logging setup
# ============================================================

# RECOMMENDED: set WANDB_API_KEY as an environment variable in your environment
# e.g. in terminal/Colab:
#   %env WANDB_API_KEY=xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
import wandb

WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")

use_wandb = WANDB_API_KEY != "" and WANDB_API_KEY != "123"

if use_wandb:
    wandb.login(key=WANDB_API_KEY)

    # turn the CFG dataclass *type* into a plain dict of its fields
    cfg_dict = {field: getattr(CFG, field) for field in CFG.__annotations__.keys()}

    wandb_run = wandb.init(
        project=CFG.project_name,
        name=CFG.run_name,
        config=cfg_dict,  
    )
    print("W&B logging enabled.")
else:
    wandb_run = None
    print("W&B logging disabled. Set WANDB_API_KEY to enable.")

In [10]:
# ============================================================
# 3. Dataset utilities: list videos & labels
# ============================================================
label2id = {"NonFight": 0, "Fight": 1}
id2label = {v: k for k, v in label2id.items()}
num_labels = len(label2id)

print("Label mapping:", label2id)

def get_video_paths(split: str):
    """
    Collect all video paths and labels for a given split: 'train' or 'val'.
    Expects:
      data_root/split/Fight/*.avi
      data_root/split/NonFight/*.avi
    """
    assert split in ["train", "val"]
    base_dir = os.path.join(CFG.data_root, split)

    video_paths = []
    labels = []

    for class_name in ["Fight", "NonFight"]:
        class_dir = os.path.join(base_dir, class_name)
        class_label = label2id[class_name]

        # support multiple video extensions just in case
        exts = ("*.avi", "*.mp4", "*.mkv", "*.mov")
        for ext in exts:
            pattern = os.path.join(class_dir, ext)
            for p in glob.glob(pattern):
                video_paths.append(p)
                labels.append(class_label)

    print(f"[{split}] Found {len(video_paths)} videos")
    return video_paths, labels

train_paths, train_labels = get_video_paths("train")
val_paths, val_labels = get_video_paths("val")

assert len(train_paths) > 0, "No training videos found. Check CFG.data_root."
assert len(val_paths) > 0, "No validation videos found. Check CFG.data_root."



Label mapping: {'NonFight': 0, 'Fight': 1}
[train] Found 1600 videos
[val] Found 400 videos


In [12]:
# ============================================================
# 4. Frame sampling + dataset class
# ============================================================
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    VideoMAEImageProcessor,
    VideoMAEForVideoClassification,
    get_cosine_schedule_with_warmup,
)

def sample_frame_indices(clip_len: int, frame_sample_rate: int, total_frames: int):
    """
    Sample `clip_len` frame indices with a given sample rate from total_frames.
    Similar strategy as used in VideoMAE and HF examples. :contentReference[oaicite:2]{index=2}
    """
    converted_len = clip_len * frame_sample_rate
    if total_frames <= converted_len:
        # just spread indices across video
        indices = np.linspace(0, total_frames - 1, clip_len).astype(np.int64)
        return np.clip(indices, 0, total_frames - 1)

    end_idx = np.random.randint(converted_len, total_frames)
    start_idx = end_idx - converted_len
    indices = np.linspace(start_idx, end_idx - 1, num=clip_len).astype(np.int64)
    return np.clip(indices, 0, total_frames - 1)

class RWF2000VideoDataset(Dataset):
    def __init__(
        self,
        video_paths: List[str],
        labels: List[int],
        image_processor: VideoMAEImageProcessor,
        is_train: bool = True,
    ):
        self.video_paths = video_paths
        self.labels = labels
        self.image_processor = image_processor
        self.is_train = is_train

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        path = self.video_paths[idx]
        label = self.labels[idx]

        # Use Decord for fast video loading
        vr = VideoReader(path, num_threads=1, ctx=cpu(0))
        total_frames = len(vr)

        indices = sample_frame_indices(
            clip_len=CFG.num_frames,
            frame_sample_rate=CFG.frame_sample_rate,
            total_frames=total_frames,
        )

        # Decord returns frames as (H, W, C) NDArray in uint8
        buffer = vr.get_batch(indices).asnumpy()  # (num_frames, H, W, C)

        # VideoMAEImageProcessor expects a list of frames (H, W, C)
        frames_list = [buffer[i] for i in range(buffer.shape[0])]

        processed = self.image_processor(
            frames_list,
            return_tensors="pt",
        )
        # pixel_values: (1, num_frames, 3, H, W)
        pixel_values = processed["pixel_values"].squeeze(0)

        return {
            "pixel_values": pixel_values,  # (num_frames, 3, H, W)
            "labels": torch.tensor(label, dtype=torch.long),
            "path": path,
        }

def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    pixel_values = torch.stack([b["pixel_values"] for b in batch])   # (B, T, C, H, W)
    labels = torch.stack([b["labels"] for b in batch])               # (B,)
    return {"pixel_values": pixel_values, "labels": labels}



2025-11-20 20:48:50.551799: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763671730.574188     153 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763671730.581042     153 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [16]:
# ============================================================
# 5. Load VideoMAE model & image processor
# ============================================================

image_processor = VideoMAEImageProcessor.from_pretrained(CFG.model_name)

model = VideoMAEForVideoClassification.from_pretrained(
    CFG.model_name,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,  # handle classification head resizing
)

# Optionally freeze early layers if you want faster training / less overfitting
# for param in model.videomae.parameters():
#     param.requires_grad = False

model.to(device)

# Dataloaders
train_dataset = RWF2000VideoDataset(
    train_paths, train_labels, image_processor=image_processor, is_train=True
)
val_dataset = RWF2000VideoDataset(
    val_paths, val_labels, image_processor=image_processor, is_train=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.train_batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    collate_fn=collate_fn,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.val_batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    collate_fn=collate_fn,
    pin_memory=True,
)

len(train_loader), len(val_loader)

Some weights of VideoMAEForVideoClassification were not initialized from the model checkpoint at MCG-NJU/videomae-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


(400, 100)

In [20]:
# ============================================================
# 6. Optimizer, scheduler & metrics
# ============================================================
import cv2
from decord import VideoReader, cpu
from tqdm import tqdm


from sklearn.metrics import f1_score

total_train_steps = len(train_loader) * CFG.num_epochs
warmup_steps = int(CFG.warmup_ratio * total_train_steps)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.learning_rate,
    weight_decay=CFG.weight_decay,
)

lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_train_steps,
)


scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda" and CFG.mixed_precision))



/tmp/ipykernel_153/1936999056.py:27: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda" and CFG.mixed_precision))


In [21]:
# ============================================================
# 7. Training & validation loop with checkpointing + logging
# ============================================================

best_val_f1 = 0.0
global_step = 0

def train_one_epoch(epoch: int):
    global global_step, best_val_f1

    model.train()
    running_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CFG.num_epochs} [train]")
    for step, batch in enumerate(pbar):
        pixel_values = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device == "cuda" and CFG.mixed_precision)):
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        lr_scheduler.step()

        running_loss += loss.item()
        global_step += 1

        if step % 10 == 0:
            avg_loss = running_loss / (step + 1)
            current_lr = lr_scheduler.get_last_lr()[0]

            pbar.set_postfix({"loss": f"{avg_loss:.4f}", "lr": f"{current_lr:.2e}"})

            if use_wandb:
                wandb.log(
                    {
                        "train/loss": avg_loss,
                        "train/lr": current_lr,
                        "train/step": global_step,
                        "epoch": epoch + 1,
                    },
                    step=global_step,
                )

def validate(epoch: int):
    global best_val_f1

    model.eval()
    val_loss = 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{CFG.num_epochs} [val]")
        for batch in pbar:
            pixel_values = batch["pixel_values"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=(device == "cuda" and CFG.mixed_precision)):
                outputs = model(pixel_values=pixel_values, labels=labels)
                loss = outputs.loss
                logits = outputs.logits

            val_loss += loss.item()

            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    avg_val_loss = val_loss / len(val_loader)

    all_preds_np = np.array(all_preds)
    all_labels_np = np.array(all_labels)
    
    accuracy = (all_preds_np == all_labels_np).mean().item()
    f1 = f1_score(all_labels_np, all_preds_np, average="macro")

    print(f"\nValidation - loss: {avg_val_loss:.4f} | acc: {accuracy:.4f} | f1: {f1:.4f}")

    if use_wandb:
        wandb.log(
            {
                "val/loss": avg_val_loss,
                "val/accuracy": accuracy,
                "val/f1": f1,
                "epoch": epoch + 1,
            },
            step=global_step,
        )

    # Checkpoint: best model (by F1)
    if f1 > best_val_f1:
        best_val_f1 = f1
        best_dir = os.path.join(CFG.output_dir, "best_model")
        os.makedirs(best_dir, exist_ok=True)

        print(f"New best F1 {best_val_f1:.4f}, saving model to {best_dir}")
        model.save_pretrained(best_dir)
        image_processor.save_pretrained(best_dir)

    # Always save "last" checkpoint for resuming
    last_dir = os.path.join(CFG.output_dir, "last")
    os.makedirs(last_dir, exist_ok=True)
    model.save_pretrained(last_dir)
    image_processor.save_pretrained(last_dir)

    return avg_val_loss, accuracy, f1

# ============================================================
# 8. Full training loop
# ============================================================

for epoch in range(CFG.num_epochs):
    train_one_epoch(epoch)
    val_loss, val_acc, val_f1 = validate(epoch)

if use_wandb:
    wandb.finish()

print("Training complete. Best F1:", best_val_f1)



Epoch 1/10 [train]:   0%|          | 0/400 [00:00<?, ?it/s]/tmp/ipykernel_153/3289318031.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda" and CFG.mixed_precision)):
Epoch 1/10 [val]:   0%|          | 0/100 [00:00<?, ?it/s]/tmp/ipykernel_153/3289318031.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda" and CFG.mixed_precision)):
Epoch 1/10 [val]: 100%|██████████| 100/100 [01:10<00:00,  1.42it/s]



Validation - loss: 0.4846 | acc: 0.7600 | f1: 0.7562
New best F1 0.7562, saving model to ./checkpoints_videomae_rwf2000/best_model


Epoch 2/10 [train]:   0%|          | 0/400 [00:00<?, ?it/s]/tmp/ipykernel_153/3289318031.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda" and CFG.mixed_precision)):
Epoch 2/10 [val]:   0%|          | 0/100 [00:00<?, ?it/s]/tmp/ipykernel_153/3289318031.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda" and CFG.mixed_precision)):
Epoch 2/10 [val]: 100%|██████████| 100/100 [00:57<00:00,  1.74it/s]



Validation - loss: 0.6468 | acc: 0.7600 | f1: 0.7478


Epoch 3/10 [train]:   0%|          | 0/400 [00:00<?, ?it/s]/tmp/ipykernel_153/3289318031.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda" and CFG.mixed_precision)):
Epoch 3/10 [val]:   0%|          | 0/100 [00:00<?, ?it/s]/tmp/ipykernel_153/3289318031.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda" and CFG.mixed_precision)):
Epoch 3/10 [val]: 100%|██████████| 100/100 [01:02<00:00,  1.61it/s]


...


Validation - loss: 0.6519 | acc: 0.8325 | f1: 0.8323


Epoch 10/10 [train]:   0%|          | 0/400 [00:00<?, ?it/s]/tmp/ipykernel_153/3289318031.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda" and CFG.mixed_precision)):
Epoch 10/10 [val]:   0%|          | 0/100 [00:00<?, ?it/s]/tmp/ipykernel_153/3289318031.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda" and CFG.mixed_precision)):
Epoch 10/10 [val]: 100%|██████████| 100/100 [00:55<00:00,  1.80it/s]



Validation - loss: 0.7101 | acc: 0.8275 | f1: 0.8269


epoch,▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▅▅▅▆▆▆▆▆▆▆▆▆▆▆▇▇▇████
train/loss,███▇▇▆▆▅▅▅▄▄▄▄▄▃▃▃▃▃▃▁▂▂▂▂▂▁▂▂▂▁▁▁▁▁▁▁▁▁
train/lr,▃▄███████▇▇▇▇▇▇▆▆▆▆▆▅▅▅▄▄▄▄▃▃▂▂▁▁▁▁▁▁▁▁▁
train/step,▁▁▁▂▂▂▂▂▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇██████
val/accuracy,▁▁▄█▃▆▇▆▆▆
val/f1,▂▁▅█▃▆▇▆▆▆
val/loss,▃▆▁▁▆▃▅█▆▇
epoch,10
train/loss,0.01497
train/lr,0.0
train/step,3991


Training complete. Best F1: 0.8569411764705883


In [ ]:
# ============================================================
# 9. Inference helper: classify a single new video
# ============================================================

def predict_video(path: str):
    model.eval()

    vr = VideoReader(path, num_threads=1, ctx=cpu(0))
    total_frames = len(vr)
    indices = sample_frame_indices(
        clip_len=CFG.num_frames,
        frame_sample_rate=CFG.frame_sample_rate,
        total_frames=total_frames,
    )
    buffer = vr.get_batch(indices).asnumpy()
    frames_list = [buffer[i] for i in range(buffer.shape[0])]

    processed = image_processor(frames_list, return_tensors="pt")
    pixel_values = processed["pixel_values"].to(device)

    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=(device == "cuda" and CFG.mixed_precision)):
            outputs = model(pixel_values=pixel_values)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1)[0]

    pred_id = int(torch.argmax(probs).item())
    pred_label = id2label[pred_id]
    confidence = float(probs[pred_id].item())

    return pred_label, confidence, probs.cpu().numpy()

# Example:
# test_path = val_paths[0]
# label, conf, probs = predict_video(test_path)
# print("Prediction:", label, "confidence:", conf)



In [22]:
import shutil
from IPython.display import FileLink

folder_path = "/kaggle/working/checkpoints_videomae_rwf2000"
zip_path = "/kaggle/working/checkpoints_videomae_rwf2000"  # without .zip

# create zip
shutil.make_archive(zip_path, 'zip', folder_path)

# show download link
FileLink('checkpoints_videomae_rwf2000.zip')


/kaggle/working/checkpoints_videomae_rwf2000.zip

In [23]:
FileLink('checkpoints_videomae_rwf2000.zip')

/kaggle/working/checkpoints_videomae_rwf2000.zip